<a href="https://colab.research.google.com/github/piyushk4732/cognitive_computing/blob/main/Assignment7_NLP_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from collections import Counter
from nltk.tokenize import RegexpTokenizer, word_tokenize
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import stopwords, wordnet

for package in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    nltk.download(package, quiet=True)

stop_words = set(stopwords.words("english"))

## 1. Preprocessing

In [ ]:
if "tickets" not in globals() or "process_ticket" not in globals():
    raise NameError("Run UCS420_LA9_Starter.ipynb first and generate your dataset.")

processed_tickets = [process_ticket(ticket) for ticket in tickets]

def get_list(result, keys):
    if isinstance(result, dict):
        for key in keys:
            if key in result:
                value = result[key]
                return value.split() if isinstance(value, str) else list(value)
    return []

rows = []
details = []
for i, (ticket, result) in enumerate(zip(tickets, processed_tickets), start=1):
    tokens = get_list(result, ["tokens", "all_tokens", "raw_tokens", "tokenized"])
    final_tokens = get_list(result, ["final_tokens", "final", "clean_tokens"])
    punctuation = [t for t in tokens if str(t) and all(c in string.punctuation for c in str(t))]
    stops = [t for t in tokens if str(t).lower() in stop_words]
    rows.append({
        "Ticket": f"Ticket {i}",
        "Number of sentences": len(nltk.sent_tokenize(ticket)),
        "Number of tokens": len(tokens),
        "Punctuation tokens": len(punctuation),
        "Stopwords": len(stops),
        "Unique tokens": len(set(str(t).lower() for t in tokens)),
        "Final tokens": len(final_tokens)
    })
    details.append((tokens, punctuation, stops, final_tokens))

stats = pd.DataFrame(rows)
totals = {"Ticket": "TOTAL"}
for col in stats.columns[1:]:
    totals[col] = stats[col].sum()
display(pd.concat([stats, pd.DataFrame([totals])], ignore_index=True))

print("Selected ticket:", tickets[0])
print("Actual tokens:", details[0][0])
print("Punctuation tokens:", details[0][1])
print("Stopwords:", details[0][2])
print("Final tokens:", details[0][3])

## 2. Regex extraction and replacement

In [ ]:
raw_text = "\n".join(tickets)

long_words = re.findall(r"\b[A-Za-z]{6,}\b", raw_text)
numbers = re.findall(r"\b\d+(?:\.\d+)*\b", raw_text)
capitalized_words = re.findall(r"\b[A-Z][a-zA-Z]*\b", raw_text)
vowel_words = re.findall(r"\b[aeiouAEIOU][A-Za-z]*\b", raw_text)

print("Words longer than 5 letters:", long_words)
print("Numbers and versions:", numbers)
print("Capitalized words:", capitalized_words)
print("Words beginning with a vowel:", vowel_words)

email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
url_pattern = r"https?://[^\s<>]+|www\.[^\s<>]+"
phone_pattern = r"(?<!\w)(?:\+?\d[\d .()\-]{7,}\d)(?!\w)"

def replace_contact_info(text):
    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(url_pattern, "<URL>", text)
    return re.sub(phone_pattern, "<PHONE>", text)

for ticket in tickets:
    print(replace_contact_info(ticket))

if "TEST_LINES" in globals():
    for line in TEST_LINES:
        print("\nTest line:", line)
        print("Long words:", re.findall(r"\b[A-Za-z]{6,}\b", line))
        print("Numbers and versions:", re.findall(r"\b\d+(?:\.\d+)*\b", line))
        print("Capitalized words:", re.findall(r"\b[A-Z][a-zA-Z]*\b", line))
        print("Vowel words:", re.findall(r"\b[aeiouAEIOU][A-Za-z]*\b", line))
        print("Replaced:", replace_contact_info(line))

## 3. Custom tokenization

In [ ]:
token_pattern = r"https?://[^\s<>]+|www\.[^\s<>]+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|\+?\d[\d .()\-]{7,}\d|\d+(?:\.\d+)+|[A-Za-z]+(?:[-'][A-Za-z]+)*|\d+|[^\w\s]"
regexp_tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    return regexp_tokenizer.tokenize(text)

if "TEST_LINES" in globals():
    for line in TEST_LINES:
        print("Text:", line)
        print("Custom:", my_tokenize(line))
        print("word_tokenize:", word_tokenize(line))

for i, ticket in enumerate(tickets, start=1):
    print(f"Ticket {i}:", my_tokenize(ticket))

### Differences between tokenizers

In [ ]:
example = "I can't connect to Wi-Fi; version 2.0.1 is installed."
print("Custom tokenizer:", my_tokenize(example))
print("word_tokenize:", word_tokenize(example))
print("1. The custom tokenizer keeps contractions such as can't together.")
print("2. It keeps hyphenated words such as Wi-Fi as one token.")
print("3. It keeps version numbers such as 2.0.1 as one token, rather than splitting at periods.")

## 4. Stemming and lemmatization

In [ ]:
porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer(r"(ing|ed|es|s)$", min=4)
lemmatizer = WordNetLemmatizer()

def to_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

final_tokens = [str(token).lower() for detail in details for token in detail[3]]
unique_final_tokens = sorted(set(final_tokens))
pos_tags = dict(nltk.pos_tag(unique_final_tokens))

stem_rows = []
for word in unique_final_tokens:
    pos = to_wordnet_pos(pos_tags[word])
    outputs = {
        "Word": word,
        "Porter Stemmer": porter.stem(word),
        "Lancaster Stemmer": lancaster.stem(word),
        "Regexp Stemmer": regexp_stemmer.stem(word),
        "WordNet Lemma": lemmatizer.lemmatize(word, pos=pos)
    }
    if len(set(outputs.values())) > 1:
        stem_rows.append(outputs)

display(pd.DataFrame(stem_rows))
print("Regexp Stemmer removes common suffixes -ing, -ed, -es, and -s. Check the actual ticket words in the table to justify which suffixes are useful for your generated dataset.")

## 5. Corpus statistics

In [ ]:
s1 = [str(token).lower() for detail in details for token in detail[0]]
s2 = [token for token in s1 if not (token and all(ch in string.punctuation for ch in token))]
s3 = [str(token).lower() for detail in details for token in detail[3]]
s4 = [porter.stem(token) for token in s3]

def corpus_stats(tokens):
    return {
        "Total tokens": len(tokens),
        "Unique tokens": len(set(tokens)),
        "TTR": len(set(tokens)) / len(tokens) if tokens else 0
    }

corpus_table = pd.DataFrame({
    "S1: All tokens": corpus_stats(s1),
    "S2: Without punctuation": corpus_stats(s2),
    "S3: Final tokens": corpus_stats(s3),
    "S4: Porter stemming": corpus_stats(s4)
}).T
display(corpus_table)

print("Top 10 words in S1:", Counter(s1).most_common(10))
print("Top 10 words in S3:", Counter(s3).most_common(10))

freq = sorted(Counter(s3).values(), reverse=True)
rank = np.arange(1, len(freq) + 1)
plt.figure(figsize=(7, 5))
plt.plot(np.log(rank), np.log(freq), marker="o", linestyle="none")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Log(rank) vs log(frequency) for S3")
plt.grid(True)
plt.show()

def make_bigrams(tokens):
    return list(zip(tokens[:-1], tokens[1:]))

print("Top 5 bigrams in S1:", Counter(make_bigrams(s1)).most_common(5))
print("Top 5 bigrams in S3:", Counter(make_bigrams(s3)).most_common(5))

## 6. Interpretation

**(a) Why does TTR change from S1 to S4?**  
TTR is the number of unique tokens divided by the total number of tokens. Preprocessing changes which tokens remain, and stemming groups related word forms into the same stem. This changes both the unique-token count and the total-token count, so TTR changes.

**(b) One difference between the tokenizers:**  
For example, `can't` may remain one token with the custom tokenizer, while `word_tokenize()` can split it. The custom regular expression explicitly preserves contractions.

**(c) One undesirable stemming result:**  
Stemming can produce a form that is not a real word or can remove too much of a word. Use an example from the differing-outputs table generated above, so the answer matches your own dataset.

**(d) POS-aware lemmatization:**  
For example, `running` can become `run` when it is tagged as a verb. Lemmatization uses grammatical context, whereas stemming applies suffix rules without considering the word's role.